# Custom Model Notebook

Use this notebook to build and train your own encoder architecture from the chunks saved by `py -3.11 -m main ingest`.

It saves the trained encoder to `models/siamese_bilstm/encoder.keras` and the vocabulary to `models/siamese_bilstm/vocab.json` so the rest of the project can keep using `main.py query`.

In [44]:
import json
import os
import sys

import numpy as np
import tensorflow as tf
from tensorflow.keras import Model, layers

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from main import load_config
from src.embedding.model import L2Normalization
from src.embedding.trainer import Trainer
from src.embedding.vocabulary import Vocabulary
from src.retrieval.vector_store import VectorStore

In [45]:
config = load_config(os.path.join(PROJECT_ROOT, 'config.yaml'))
config

{'pdf_directory': 'data/pdfs',
 'chunk_size': 300,
 'chunk_overlap': 50,
 'chunk_strategy': 'sentence',
 'nltk_data_path': None,
 'remove_stopwords': True,
 'apply_lemmatization': True,
 'apply_pos_tagging': True,
 'encoder_architecture': 'bilstm',
 'embedding_dim': 128,
 'lstm_units': 64,
 'dense_units': 128,
 'max_sequence_length': 200,
 'vocab_size': 20000,
 'batch_size': 32,
 'epochs': 100,
 'model_save_path': 'models/siamese_bilstm',
 'min_training_window_words': 20,
 'min_training_window_stride': 10,
 'top_k': 5,
 'similarity_threshold': 0.3,
 'vector_store_path': 'data/vectors',
 'processed_chunks_path': 'data/processed/chunks.json',
 'enable_query_rewriting': True,
 'max_synonyms': 3,
 'mode': 'extractive',
 'llm_provider': None,
 'llm_api_key': None,
 'max_context_tokens': 4000,
 'eval_output_dir': 'evaluation_results'}

## Load processed chunks

Make sure you have already run:

`py -3.11 -m main ingest`

In [46]:
chunks_path = os.path.join(PROJECT_ROOT, config['processed_chunks_path'])
with open(chunks_path, 'r', encoding='utf-8') as f:
    chunks = json.load(f)

trainer = Trainer(config)
training_chunks = trainer._expand_chunks_for_training(chunks)
print('Original chunks:', len(chunks))
print('Training chunks:', len(training_chunks))
training_chunks[:2]

Original chunks: 7
Training chunks: 7


[{'text': 'Introduction to AI\nArtificial Intelligence (AI) is the simulation of human intelligence by machines. It seeks to create system\nArtificial Intelligence (AI) is the simulation of human intelligence by machines. It seeks to create system\nArtificial Intelligence (AI) is the simulation of human intelligence by machines. It seeks to create system\nArtificial Intelligence (AI) is the simulation of human intelligence by machines. It seeks to create system\nArtificial Intelligence (AI) is the simulation of human intelligence by machines. It seeks to create system\nArtificial Intelligence (AI) is the simulation of human intelligence by machines. It seeks to create system\nArtificial Intelligence (AI) is the simulation of human intelligence by machines. It seeks to create system\nArtificial Intelligence (AI) is the simulation of human intelligence by machines. It seeks to create system\nArtificial Intelligence (AI) is the simulation of human intelligence by machines. It seeks to cre

## Build training pairs

In [47]:
texts_a, texts_b, labels = trainer.generate_pairs(chunks)
print('Pairs:', len(labels), 'Positive:', sum(labels), 'Negative:', len(labels) - sum(labels))
assert labels, 'Not enough training pairs. Add more PDFs or lower chunk_size in config.yaml.'

Pairs: 12 Positive: 6 Negative: 6


## Build vocabulary and encode text

In [48]:
vocab = Vocabulary(max_size=config.get('vocab_size', 20000))
vocab.build([chunk.get('processed_text', chunk['text']) for chunk in chunks])

max_len = config.get('max_sequence_length', 200)
seqs_a = np.array([vocab.encode(text, max_len) for text in texts_a])
seqs_b = np.array([vocab.encode(text, max_len) for text in texts_b])
labels_arr = np.array(labels, dtype=np.float32)

seqs_a.shape, seqs_b.shape, labels_arr.shape

  Vocabulary built: 62 tokens


((12, 200), (12, 200), (12,))

## Define your own encoder

Edit this cell to make your own model. The only requirement is that it returns one vector per input sequence.

In [49]:
def build_custom_encoder(vocab_size, embedding_dim, hidden_units, dense_units, max_seq_length):
    inputs = layers.Input(shape=(max_seq_length,), name='text_input')
    x = layers.Embedding(vocab_size, embedding_dim, name='embedding')(inputs)
    x = layers.Bidirectional(layers.GRU(hidden_units, return_sequences=True), name='bigru')(x)
    x = layers.GlobalMaxPooling1D(name='pool')(x)
    x = layers.Dense(dense_units, activation='relu', name='dense')(x)
    outputs = L2Normalization(name='l2_normalize')(x)
    return Model(inputs, outputs, name='custom_text_encoder')

encoder = build_custom_encoder(
    vocab_size=vocab.size,
    embedding_dim=config.get('embedding_dim', 128),
    hidden_units=config.get('lstm_units', 64),
    dense_units=config.get('dense_units', 128),
    max_seq_length=max_len,
)
encoder.summary()

Model: "custom_text_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_input (InputLayer)         │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 200, 128)       │         7,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bigru (Bidirectional)           │ (None, 200, 128)       │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool (GlobalMaxPooling1D)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ l2_normalize (L2Normalization)  │ (None, 128)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 98,944 (386.50 KB)

 Trainable params: 98,944 (386.50 KB)

 Non-trainable params: 0 (0.00 B)

## Wrap it in a Siamese model

In [50]:
input_a = layers.Input(shape=(max_len,), name='input_a')
input_b = layers.Input(shape=(max_len,), name='input_b')
encoded_a = encoder(input_a)
encoded_b = encoder(input_b)
similarity = layers.Dot(axes=1, normalize=False, name='cosine_similarity')([encoded_a, encoded_b])
siamese = Model([input_a, input_b], similarity, name='custom_siamese')

def contrastive_loss(y_true, y_pred, margin=0.5):
    y_true = tf.cast(y_true, tf.float32)
    return tf.reduce_mean(
        y_true * tf.square(1 - y_pred) + (1 - y_true) * tf.square(tf.maximum(y_pred - margin, 0))
    )

siamese.compile(optimizer='adam', loss=contrastive_loss, metrics=['accuracy'])
siamese.summary()

Model: "custom_siamese"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_a             │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_b             │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ custom_text_encoder │ (None, 128)       │     98,944 │ input_a[0][0],    │
│ (Functional)        │                   │            │ input_b[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cosine_similarity   │ (None, 1)         │          0 │ custom_text_enco… │
│ (Dot)               │                   │            │ custom_text_enco… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 98,944 (386.50 KB)

 Trainable params: 98,944 (386.50 KB)

 Non-trainable params: 0 (0.00 B)

## Train

In [51]:
validation_split = 0.2 if len(labels_arr) >= 5 else 0.0
history = siamese.fit(
    [seqs_a, seqs_b],
    labels_arr,
    batch_size=min(config.get('batch_size', 32), len(labels_arr)),
    epochs=config.get('epochs', 200),
    validation_split=validation_split,
)
history.history

Epoch 1/100


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.4444 - loss: 0.1071 - val_accuracy: 0.6667 - val_loss: 0.0232
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.4444 - loss: 0.0655 - val_accuracy: 0.6667 - val_loss: 0.0491
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.4444 - loss: 0.0294 - val_accuracy: 0.6667 - val_loss: 0.0838
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.8889 - loss: 0.0207 - val_accuracy: 0.6667 - val_loss: 0.0863
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.5556 - loss: 0.0133 - val_accuracy: 0.6667 - val_loss: 0.0826
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - accuracy: 0.5556 - loss: 0.0123 - val_accuracy: 0.6667 - val_loss: 0.0796
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.6667 - loss: 0.0109 - val_accuracy: 0.6667 - val_loss: 0.0790
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.7778 - loss: 0.0109 - val_accuracy: 0.6667 - val_loss: 0.0789
Epoch

{'accuracy': [0.4444444477558136,
  0.4444444477558136,
  0.4444444477558136,
  0.8888888955116272,
  0.5555555820465088,
  0.5555555820465088,
  0.6666666865348816,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.6666666865348816,
  0.6666666865348816,
  0.6666666865348816,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.6666666865348816,
  0.6666666865348816,
  0.6666666865348816,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.7777777910232544,
  0.6666666865348816

## Save encoder and rebuild the vector store

In [52]:
model_dir = os.path.join(PROJECT_ROOT, config['model_save_path'])
os.makedirs(model_dir, exist_ok=True)

encoder_path = os.path.join(model_dir, 'encoder.keras')
vocab_path = os.path.join(model_dir, 'vocab.json')
encoder.save(encoder_path)
vocab.save(vocab_path)

chunk_sequences = np.array([
    vocab.encode(chunk.get('processed_text', chunk['text']), max_len) for chunk in chunks
])
vectors = encoder.predict(chunk_sequences, verbose=0)
store = VectorStore(os.path.join(PROJECT_ROOT, config['vector_store_path']))
store.add_all(chunks, vectors)
store.save()

print('Saved encoder to', encoder_path)
print('Saved vocabulary to', vocab_path)
print('Rebuilt vector store at', os.path.join(PROJECT_ROOT, config['vector_store_path']))

Saved encoder to d:\SIC\models/siamese_bilstm\encoder.keras
Saved vocabulary to d:\SIC\models/siamese_bilstm\vocab.json
Rebuilt vector store at d:\SIC\data/vectors


## Next step

After you run the save cell above, query the project normally:

`py -3.11 -m main query "your question"`